## 02_silver.ipynb — build silver from bronze.

> Steps: enrich (canonicalize) -> DQ filter (high-confidence + mapped) -> dedupe.
> Failed filter -> output/silver_quarantine.csv (nothing dropped silently).

>Dedupes on company + period + metric_canonical + value, so identical
re-ingested rows collapse while corrections (different value) are retained.
When a later report revises a figure, silver holds both the original and the
corrected row; ingested_at is the tiebreaker for which is current, used by
gold / silver-upper.

In [ ]:


import json
from pathlib import Path
import pandas as pd
 
from canonical import canonicalize, CANONICAL_METRICS
 
BRONZE_DIR = Path("output/bronze")
OUT_DIR = Path("output/silver")
 
 
# ---------- load ----------
 
def load_bronze():
    records = []
    for f in sorted(BRONZE_DIR.glob("*.jsonl")):
        with open(f, encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    return records
 
 
# ---------- enrich ----------
 
def enrich(record):
    r = canonicalize(record["metric"])
    canonical = r["canonical"]
    unit = CANONICAL_METRICS[canonical]["unit"] if canonical else record.get("unit")
    return {**record,
            "metric_canonical": r["canonical"],
            "unit": unit, 
            "match_method": r["method"],
            "match_confidence": r["confidence"]}

 
# ---------- build ----------
 
def build_silver(bronze_records):
    enriched = [enrich(r) for r in bronze_records]
    df = pd.DataFrame(enriched)
 
    # normalize the extraction-confidence column name (older bronze used "confidence")
    if "extraction_confidence" not in df.columns and "confidence" in df.columns:
        df = df.rename(columns={"confidence": "extraction_confidence"})
    if "extraction_confidence" not in df.columns:
        df["extraction_confidence"] = "high"   # default if bronze didn't tag it
 
    # filter: trustworthy AND mapped
    keep = (
        (df["extraction_confidence"] == "high")
        & (df["match_confidence"] == "high")
        & df["metric_canonical"].notna()
    )
    silver = df[keep].copy()
    silver_quarantine = df[~keep].copy()
 
    # dedup: restatement-aware. Prefer later-published report; fall back to ingest time.
    sort_keys = [k for k in ["report_period", "ingested_at"] if k in silver.columns]
    if sort_keys:
        silver = silver.sort_values(sort_keys)
    silver = silver.drop_duplicates(
        ["company", "period", "metric_canonical","value"], keep="last"
    ).reset_index(drop=True)
 
    return silver, silver_quarantine
 
 
# ---------- driver ----------
 
def main():
    bronze = load_bronze()
    print(f"bronze:   {len(bronze)} records")
 
    silver, silver_quarantine = build_silver(bronze)
    print(f"silver:   {len(silver)} kept")
    print(f"quarantine: {len(silver_quarantine)} (low extraction / low match / unmapped)")
 
    OUT_DIR.mkdir(exist_ok=True)
    silver.to_csv(OUT_DIR / "silver.csv", index=False)
    silver_quarantine.to_csv(OUT_DIR / "silver_quarantine.csv", index=False)
 
    if len(silver_quarantine):
        print("\nsilver_quarantine rows:")
        for _, r in silver_quarantine.iterrows():
            if pd.isna(r["metric_canonical"]):
                reason = "unmapped"
            elif r["extraction_confidence"] != "high":
                reason = "low extraction"
            else:
                reason = f"{r['match_confidence']} match"
            print(f"  {r['company']:12s} {r['metric']:34s} [{reason}]")
 
    print("\nsilver preview:")
    cols = ["company", "period", "metric_canonical", "value", "unit"]
    cols = [c for c in cols if c in silver.columns]
    print(silver[cols].to_string(index=False))
 
    print(f"\nwrote {OUT_DIR/'silver.csv'} and {OUT_DIR/'silver_quarantine.csv'}")
 
 
if __name__ == "__main__":
    main()
 